# Meta-Governance Layer — Dev Log

## Objetivo

Governança SOBRE governança: audita se cada nó de uma federação está
executando seus próprios checks de saúde corretamente — complementar a
`federated_governance` (V2), que agrega métricas de DECISÃO, não de
compliance de processo.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import tempfile
from pathlib import Path
from core.incident_response.log import IncidentLog
from core.self_healing_governance.healer import check_and_heal
from core.meta_governance.audit import audit_federation_health

demo_dir = Path(tempfile.mkdtemp(prefix="meta_gov_demo_"))
incident_log = IncidentLog(storage_path=demo_dir / "incidents.json")

node_a = check_and_heal({"audit_chain_integrity": True, "prompt_security_coverage": True}, incident_log=incident_log)
node_b = check_and_heal({"audit_chain_integrity": False, "prompt_security_coverage": True}, incident_log=incident_log)

node_health = {
    "filial_sp": {a.check_name: a.healthy for a in node_a},
    "filial_rj": {a.check_name: a.healthy for a in node_b},
}
result = audit_federation_health(node_health)
for n in result.per_node:
    print(f"  {n.node_id}: {n.checks_healthy}/{n.checks_total} saudáveis ({n.compliance_rate:.0%}) | compliant={n.compliant}")
print()
print(result.summary)

  filial_sp: 2/2 saudáveis (100%) | compliant=True
  filial_rj: 1/2 saudáveis (50%) | compliant=False

Federação com 2 nó(s), taxa agregada 75%. 1 nó(s) ABAIXO do limiar de 100%: filial_rj.


## Testes e Handoff

```
"C:/Users/Yuri_/.venvs/athenagov-ai/Scripts/python.exe" -m pytest core/meta_governance/tests -v
```

8/8 testes passando, incluindo integração real com `self_healing_governance`.